# Transaction Summary

**Business question:** What products sell the most, how are they paid for, and which combinations drive the most revenue?

This notebook aggregates the bakehouse sales transaction data to answer:
- Which products generate the highest revenue?
- What share of total revenue does each product contribute?
- How do customers prefer to pay, and does it vary by product?
- Which product × payment-method combinations are most valuable?

**Output table:** `{catalog}.{schema}.transaction_summary`  
**Source table:** `samples.bakehouse.sales_transactions`

In [ ]:
dbutils.widgets.text("catalog", "dev_databricks_analytics")
dbutils.widgets.text("schema", "md")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"Target: {catalog}.{schema}")

## Step 1 — Explore raw transactions

Preview the data to confirm column names and check for NULLs before transforming.

In [ ]:
transactions = spark.read.table("samples.bakehouse.sales_transactions")
print(f"Transactions: {transactions.count():,} rows")
transactions.printSchema()
transactions.show(5, truncate=False)

## Step 2 — Product revenue with percentage share

We compute each product's total revenue and its **percentage share of overall revenue**
using a window function. This avoids a subquery to get the grand total and keeps
the aggregation in a single pass.

`SUM(total_revenue) OVER ()` with no PARTITION BY is an unbounded window over
the entire result set — this is the standard SQL pattern for computing ratios
against a grand total.

In [ ]:
product_revenue = spark.sql("""
    WITH product_agg AS (
        SELECT
            product,
            COUNT(*)                                          AS total_transactions,
            SUM(quantity)                                     AS total_quantity,
            ROUND(SUM(totalPrice), 2)                         AS total_revenue,
            ROUND(AVG(totalPrice), 2)                         AS avg_transaction_value,
            ROUND(AVG(totalPrice / NULLIF(quantity, 0)), 2)   AS avg_price_per_unit
        FROM samples.bakehouse.sales_transactions
        GROUP BY product
    )
    SELECT
        product,
        total_transactions,
        total_quantity,
        total_revenue,
        avg_transaction_value,
        avg_price_per_unit,
        -- Window function: share of total revenue with no partition = entire dataset
        ROUND(
            total_revenue * 100.0
            / SUM(total_revenue) OVER (),
        2)                                                    AS revenue_share_pct,
        DENSE_RANK() OVER (ORDER BY total_revenue DESC)       AS revenue_rank
    FROM product_agg
    ORDER BY revenue_rank
""")
product_revenue.show(truncate=False)

## Step 3 — Payment method breakdown

Understanding how customers pay helps with payment processing costs and
promotional strategy (e.g. discounting cash payments to shift to lower-cost
digital methods).

In [ ]:
payment_breakdown = spark.sql("""
    SELECT
        paymentMethod,
        COUNT(*)                                              AS total_transactions,
        ROUND(SUM(totalPrice), 2)                             AS total_revenue,
        ROUND(AVG(totalPrice), 2)                             AS avg_transaction_value,
        ROUND(
            SUM(totalPrice) * 100.0 / SUM(SUM(totalPrice)) OVER (),
        2)                                                    AS revenue_share_pct
    FROM samples.bakehouse.sales_transactions
    GROUP BY paymentMethod
    ORDER BY total_revenue DESC
""")
payment_breakdown.show(truncate=False)

## Step 4 — Product × payment method revenue matrix

The cross-tab view answers: "For each product, which payment method do customers
prefer, and does that vary by product?"

This is the core of the `transaction_summary` output table — combining both
dimensions gives analysts the most flexible aggregation to slice.

In [ ]:
transaction_summary = spark.sql("""
    SELECT
        product,
        paymentMethod,
        COUNT(*)                                              AS total_transactions,
        SUM(quantity)                                         AS total_quantity,
        ROUND(SUM(totalPrice), 2)                             AS total_revenue,
        ROUND(AVG(totalPrice), 2)                             AS avg_transaction_value,
        ROUND(AVG(totalPrice / NULLIF(quantity, 0)), 2)       AS avg_price_per_unit,
        -- Rank payment methods within each product to find most popular per product
        DENSE_RANK() OVER (
            PARTITION BY product
            ORDER BY SUM(totalPrice) DESC
        )                                                     AS payment_rank_within_product
    FROM samples.bakehouse.sales_transactions
    GROUP BY product, paymentMethod
    ORDER BY product, payment_rank_within_product
""")
transaction_summary.show(50, truncate=False)

## Step 5 — Write to Unity Catalog

In [ ]:
transaction_summary.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.transaction_summary")
print(f"Written {transaction_summary.count()} rows to {catalog}.{schema}.transaction_summary")